In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import IngestionAgent

data_root = project_root / 'data' / 'v5.2'
agent = IngestionAgent(data_root=data_root)

print('Project root:', project_root)
print('Data root:', data_root)
print('Available datasets:', agent.available_datasets())
print('Feature subsets:', agent.feature_subset_names())

for dataset_name in ('train', 'validation', 'live'):
    summary = agent.summarize_dataset(
        dataset_name,
        feature_subset='medium',
        include_targets=dataset_name in {'train', 'validation'},
    )
    preview_columns = list(summary.columns[:8])
    print(f'{dataset_name}: rows={summary.row_count:,} columns={len(summary.columns)} path={summary.path.name}')
    print('  preview columns:', preview_columns)
    print('  preview schema:', {name: summary.schema[name] for name in preview_columns})

In [ ]:
from src.features import PurgedEraSplitter

splitter = PurgedEraSplitter(n_splits=5, purge_buffer=4)
train_lazy = agent.scan_dataset('train', include_metadata=True)
splits = splitter.split(train_lazy)

print('Fold | Training Eras | Validation Eras | Zero Leakage')
print('-----|---------------|-----------------|-------------')
for fold_number, (train_eras, validation_eras) in enumerate(splits, start=1):
    train_ordinals = [int(era) for era in train_eras]
    validation_ordinals = [int(era) for era in validation_eras]
    zero_leakage = all(abs(train_era - valid_era) > splitter.purge_buffer for train_era in train_ordinals for valid_era in validation_ordinals)
    print(f'{fold_number:>4} | {len(train_eras):>13} | {len(validation_eras):>15} | {zero_leakage}')